<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk10_mostlyai_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# wk10 — MostlyAI Verification and Conditional Cold Generation

**Purpose.** Establish the authoritative MostlyAI numbers with full provenance, and
test whether *conditionally generated* Cold rows change the outcome.

**Why this notebook exists.** `wk4_mostlyai_autotherm.ipynb` used
`mostly.probe(g, size=...)`, which is unconditional generation: the synthetic label
distribution mirrors the real one, so Cold stayed at ~4.5% and augmentation added a
proportional copy of the whole distribution rather than oversampling the minority.
The claim "synthetic Cold data cannot rescue the Cold class" was therefore never
directly tested for this generator.




## Part 0 — Setup and provenance manifest

In [1]:
!pip install -q -U "mostlyai[local]" datasets 2>&1 | tail -3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.4 MB/s eta 0:00:00


In [2]:
import os, re, json, platform, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)

RANDOM_STATE = 42
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

MANIFEST = {
    "run_id": RUN_ID,
    "started": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "random_state": RANDOM_STATE,
}

# Collected as we go; written to disk at the end.
RESULTS = []

def log_result(method, protocol, granularity, y_true, y_pred, notes=""):
    """Single point of truth. Every number in the thesis comes from here."""
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal   = balanced_accuracy_score(y_true, y_pred)
    per   = f1_score(y_true, y_pred, average=None,
                     labels=sorted(pd.unique(y_true)), zero_division=0)
    labels = sorted(pd.unique(y_true))
    cold_label = min(labels)
    cold_f1 = dict(zip(labels, per))[cold_label]

    row = {
        "run_id": RUN_ID, "method": method, "protocol": protocol,
        "granularity": granularity, "macro_f1": round(macro, 4),
        "balanced_acc": round(bal, 4), "cold_f1": round(cold_f1, 4),
        "n_test": len(y_true), "notes": notes,
    }
    for lb, f in zip(labels, per):
        row[f"f1_class_{lb}"] = round(f, 4)
    RESULTS.append(row)

    print(f"  {method:28} {protocol:10} {granularity}-class  "
          f"macro_f1={macro:.4f}  cold_f1={cold_f1:.4f}")
    return row

print("Run ID:", RUN_ID)
print(json.dumps(MANIFEST, indent=2))

Run ID: 20260804_231218
{
  "run_id": "20260804_231218",
  "started": "2026-08-04T23:12:18",
  "python": "3.12.13",
  "numpy": "2.0.2",
  "pandas": "2.2.2",
  "sklearn": "1.6.1",
  "random_state": 42
}


## Part 1 — Data, split, and features

This reproduces `wk4` exactly.

In [3]:
from datasets import load_dataset

print("Loading AutoTherm indoor...")
dataset  = load_dataset("kopetri/AutoTherm", "indoor")
train_df = dataset["train"].to_pandas()

def extract_participant_id(filename):
    m = re.search(r"participant_\d+", filename)
    return m.group() if m else "unknown"

train_df["participant_id"] = train_df["file_name"].apply(extract_participant_id)
train_df["Label_3class"]   = train_df["Label"].apply(
    lambda x: -1 if x <= -2 else (0 if x <= 1 else 1)
)

TEST_PARTICIPANTS = ["participant_14", "participant_16", "participant_20"]
train_split = train_df[~train_df["participant_id"].isin(TEST_PARTICIPANTS)]
test_split  = train_df[ train_df["participant_id"].isin(TEST_PARTICIPANTS)]

MANIFEST.update({
    "dataset": "kopetri/AutoTherm indoor",
    "test_participants": TEST_PARTICIPANTS,
    "n_train_rows": int(len(train_split)),
    "n_test_rows": int(len(test_split)),
    "n_train_participants": int(train_split["participant_id"].nunique()),
})

print(f"Train: {len(train_split):,} rows, "
      f"{train_split['participant_id'].nunique()} participants")
print(f"Test:  {len(test_split):,} rows, "
      f"{test_split['participant_id'].nunique()} participants")
print("\nTest label distribution (7-class):")
print(test_split["Label"].value_counts().sort_index())

Loading AutoTherm indoor...


README.md:   0%|          | 0.00/8.57k [00:00<?, ?B/s]

indoor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 29.8MB            

indoor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

...

indoor/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.41MB            

indoor/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1566728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

Train: 1,276,709 rows, 13 participants
Test:  290,019 rows, 3 participants

Test label distribution (7-class):
Label
-3    22556
-2    19285
-1    85260
 0    39044
 1    16824
 2    51786
 3    55264
Name: count, dtype: int64


In [4]:
# Columns excluded from the feature matrix.
# NOTE: this is the pre-defined drop list that caused BUG-2/BUG-3. Any column
# derived from the target MUST appear here, or it leaks.
DROP_COLS = [
    "file_name", "Timestamp", "participant_id",
    "Air-Velocity", "Metabolic-Rate",
    "Nose", "Neck", "RShoulder", "RElbow",
    "LShoulder", "LElbow", "REye", "LEye", "REar", "LEar",
    "Emotion-Self", "Emotion-ML",
    "Label", "Label_3class",          # both targets, always dropped
]

def prepare_features(df, target_col):
    """Returns (X, y). Drops DROP_COLS plus anything target-derived."""
    X = df.drop(columns=[c for c in DROP_COLS if c in df.columns],
                errors="ignore").copy()
    if "Gender" in X.columns:
        X["Gender"] = LabelEncoder().fit_transform(X["Gender"].astype(str))
    X = X.select_dtypes(include=[np.number])
    y = df[target_col]
    return X, y

X_train_7, y_train_7 = prepare_features(train_split, "Label")
X_test_7,  y_test_7  = prepare_features(test_split,  "Label")
X_train_3, y_train_3 = prepare_features(train_split, "Label_3class")
X_test_3,  y_test_3  = prepare_features(test_split,  "Label_3class")

FEATURE_COLS = list(X_train_7.columns)
MANIFEST["feature_cols"] = FEATURE_COLS
MANIFEST["n_features"] = len(FEATURE_COLS)

# LEAKAGE ASSERTIONS. These must pass. If one fails, the results are invalid.
assert "Label" not in X_train_7.columns, "LEAK: Label in features"
assert "Label_3class" not in X_train_7.columns, "LEAK: Label_3class in features"
assert list(X_train_7.columns) == list(X_test_7.columns), "train/test column mismatch"
assert set(train_split["participant_id"]) & set(test_split["participant_id"]) == set(), \
    "LEAK: participant appears in both train and test"

print(f"{len(FEATURE_COLS)} features:")
print(FEATURE_COLS)
print("\nAll leakage assertions passed.")

18 features:
['Age', 'Gender', 'Weight', 'Height', 'Bodyfat', 'Bodytemp', 'Sport-Last-Hour', 'Time-Since-Meal', 'Tiredness', 'Clothing-Level', 'Radiation-Temp', 'PCE-Ambient-Temp', 'Wrist_Skin_Temperature', 'Heart_Rate', 'GSR', 'Ambient_Temperature', 'Ambient_Humidity', 'Solar_Radiation']

All leakage assertions passed.


## Part 2 — Baseline

These two numbers anchor everything else. They must come out as 0.2858 and 0.7163.
If they do not, something upstream has changed and no comparison below is valid.

In [5]:
print("Training baselines...")

clf_base_7 = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
clf_base_7.fit(X_train_7, y_train_7)
base7 = log_result("Baseline (real only)", "TRTR", 7,
                   y_test_7, clf_base_7.predict(X_test_7))

clf_base_3 = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
clf_base_3.fit(X_train_3, y_train_3)
base3 = log_result("Baseline (real only)", "TRTR", 3,
                   y_test_3, clf_base_3.predict(X_test_3))

BASE_7, BASE_3 = base7["macro_f1"], base3["macro_f1"]

print("\n--- REPRODUCIBILITY CHECK vs wk4 ---")
for name, got, want in [("7-class", BASE_7, 0.2858), ("3-class", BASE_3, 0.7163)]:
    status = "MATCH" if abs(got - want) < 0.001 else "*** MISMATCH — INVESTIGATE ***"
    print(f"  {name}: got {got:.4f}, wk4 reported {want:.4f}  -> {status}")

Training baselines...
  Baseline (real only)         TRTR       7-class  macro_f1=0.2858  cold_f1=0.0000
  Baseline (real only)         TRTR       3-class  macro_f1=0.7163  cold_f1=0.7120

--- REPRODUCIBILITY CHECK vs wk4 ---
  7-class: got 0.2858, wk4 reported 0.2858  -> MATCH
  3-class: got 0.7163, wk4 reported 0.7163  -> MATCH


## Part 3 — MostlyAI generator

Trains on the same 100k stratified sample used in `wk4`, so the comparison is like
for like. Training takes several minutes.

In [6]:
from mostlyai.sdk import MostlyAI

SDV_DROP = [
    "file_name", "Timestamp", "participant_id",
    "Air-Velocity", "Metabolic-Rate",
    "Nose", "Neck", "RShoulder", "RElbow",
    "LShoulder", "LElbow", "REye", "LEye", "REar", "LEar",
    "Emotion-Self", "Emotion-ML",
    "Label_3class",           # Label is KEPT: it is what we generate
]

mostly_train = train_split.drop(columns=SDV_DROP, errors="ignore").copy()
mostly_train["Gender"] = LabelEncoder().fit_transform(mostly_train["Gender"].astype(str))

SAMPLE_SIZE = 100_000
mostly_train_sample = (
    mostly_train.groupby("Label", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), int(SAMPLE_SIZE * len(x) / len(mostly_train))),
        random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

MANIFEST.update({
    "mostlyai_sample_size": int(len(mostly_train_sample)),
    "mostlyai_sample_label_dist":
        {str(k): int(v) for k, v in
         mostly_train_sample["Label"].value_counts().sort_index().items()},
})

print(f"Training sample: {len(mostly_train_sample):,} rows")
print(mostly_train_sample["Label"].value_counts().sort_index())

Training sample: 99,996 rows
Label
-3     3761
-2     9151
-1    20622
 0    24056
 1    13968
 2    15896
 3    12542
Name: count, dtype: int64


/tmp/ipykernel_6668/1246439729.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

In [7]:
mostly = MostlyAI(local=True)
print("Training MostlyAI generator... (several minutes)")
g = mostly.train(name=f"autotherm_indoor_{RUN_ID}", data=mostly_train_sample)
print("Generator trained.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Initializing Synthetic Data SDK 6.1.1 in LOCAL mode 🏠

Connected to ]8;id=482145;file:///root/mostlyai\/root/]8;;\]8;id=96180;file:///root/mostlyai\mostlyai]8;;\ with 13 GB RAM, 2 CPUs, 0 GPUs available

Training MostlyAI generator... (several minutes)


Created generator b2cf35b1-d130-4276-b0d3-9a8842518844

Started generator training

Output()

🎉 Your generator is ready! Use it to create synthetic data. Publish it so others can do the same.

Generator trained.


## Part 4 — Condition A: unconditional generation

This reproduces exactly what `wk4` did. Expected: TSTR 3-class 0.7003,
Augmented 3-class 0.7066, both below the 0.7163 baseline, Cold F1 zero throughout.

Note the printed Cold share of the synthetic data. It will be close to the real
share, which is the point: **this is not minority oversampling.**

In [8]:
print("Generating unconditional synthetic data...")
synth_uncond = mostly.probe(g, size=len(mostly_train_sample))

real_cold = (mostly_train_sample["Label"] == -3).mean()
syn_cold  = (synth_uncond["Label"] == -3).mean()

MANIFEST.update({
    "uncond_n_synthetic": int(len(synth_uncond)),
    "uncond_real_cold_share": round(float(real_cold), 4),
    "uncond_synth_cold_share": round(float(syn_cold), 4),
})

print(f"Generated {len(synth_uncond):,} rows")
print(f"Cold share  real: {real_cold:.4f}   synthetic: {syn_cold:.4f}")
print("\n>>> If these two are similar, augmentation did NOT oversample Cold. <<<")

Generating unconditional synthetic data...
Generated 99,996 rows
Cold share  real: 0.0376   synthetic: 0.0354

>>> If these two are similar, augmentation did NOT oversample Cold. <<<


In [9]:
def prep_synth(df, target_col):
    """Align synthetic frame to the real feature matrix. Column order matters."""
    d = df.copy()
    if target_col == "Label_3class" and "Label_3class" not in d.columns:
        d["Label_3class"] = d["Label"].apply(
            lambda x: -1 if x <= -2 else (0 if x <= 1 else 1))
    if "Gender" in d.columns and d["Gender"].dtype == object:
        d["Gender"] = LabelEncoder().fit_transform(d["Gender"].astype(str))
    X = d.reindex(columns=FEATURE_COLS)
    X = X.apply(pd.to_numeric, errors="coerce")
    y = d[target_col]
    keep = X.notna().all(axis=1)
    return X[keep].reset_index(drop=True), y[keep].reset_index(drop=True)


def evaluate(synth_df, tag, notes=""):
    """TSTR and Augmented, at both granularities. Everything goes through log_result."""
    for gran, tcol, Xtr, ytr, Xte, yte in [
        (7, "Label",        X_train_7, y_train_7, X_test_7, y_test_7),
        (3, "Label_3class", X_train_3, y_train_3, X_test_3, y_test_3),
    ]:
        Xs, ys = prep_synth(synth_df, tcol)
        if ys.nunique() < 2:
            print(f"  [skip] {tag} {gran}-class: synthetic data has <2 classes")
            continue

        clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
        clf.fit(Xs, ys)
        log_result(tag, "TSTR", gran, yte, clf.predict(Xte), notes)

        Xa = pd.concat([Xtr.reset_index(drop=True), Xs], axis=0).reset_index(drop=True)
        ya = pd.concat([ytr.reset_index(drop=True), ys], axis=0).reset_index(drop=True)
        clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
        clf.fit(Xa, ya)
        log_result(tag, "Augmented", gran, yte, clf.predict(Xte),
                   notes + f" | synth={len(Xs)} ({100*len(Xs)/len(Xa):.1f}% of train)")


print("CONDITION A — unconditional generation (reproduces wk4)\n")
evaluate(synth_uncond, "MostlyAI", notes="unconditional probe")

CONDITION A — unconditional generation (reproduces wk4)

  MostlyAI                     TSTR       7-class  macro_f1=0.2119  cold_f1=0.0000
  MostlyAI                     Augmented  7-class  macro_f1=0.2563  cold_f1=0.0000
  MostlyAI                     TSTR       3-class  macro_f1=0.6601  cold_f1=0.7341
  MostlyAI                     Augmented  3-class  macro_f1=0.6719  cold_f1=0.7440


## Part 5 — Condition B: conditional Cold generation

**This is the experiment `wk4` never ran.**

Here we seed the generator with `Label = -3` so it produces Cold rows specifically,
then augment the real training set with them. This directly tests the claim your
thesis makes: that no quantity of synthetic Cold data recovers the class.

Three augmentation levels are tested, deliberately stopping short of full parity.
Full rebalancing causes prior shift, which degrades calibration and is not the
recommended practice.

In [10]:
N_COLD_TARGETS = [10_000, 50_000, 200_000]

real_cold_n = int((y_train_7 == -3).sum())
print(f"Real Cold rows in training set: {real_cold_n:,} "
      f"({100*real_cold_n/len(y_train_7):.2f}%)\n")

cold_synth_sets = {}
for n in N_COLD_TARGETS:
    print(f"Generating {n:,} conditional Cold rows...")
    seed = pd.DataFrame({"Label": [-3] * n})
    try:
        sc = mostly.probe(g, seed=seed)
    except TypeError:
        # Older SDK signature
        sc = mostly.probe(g, size=n, seed=seed)
    got_cold = int((sc["Label"] == -3).sum())
    print(f"  returned {len(sc):,} rows, {got_cold:,} labelled Cold "
          f"({100*got_cold/max(len(sc),1):.1f}%)")
    cold_synth_sets[n] = sc

MANIFEST["conditional_cold_targets"] = N_COLD_TARGETS
MANIFEST["real_cold_n"] = real_cold_n

Real Cold rows in training set: 48,020 (3.76%)

Generating 10,000 conditional Cold rows...
  returned 10,000 rows, 10,000 labelled Cold (100.0%)
Generating 50,000 conditional Cold rows...
  returned 50,000 rows, 50,000 labelled Cold (100.0%)
Generating 200,000 conditional Cold rows...
  returned 200,000 rows, 200,000 labelled Cold (100.0%)


In [11]:
print("CONDITION B — conditional Cold augmentation\n")

for n, sc in cold_synth_sets.items():
    Xs, ys = prep_synth(sc, "Label")
    if len(Xs) == 0:
        print(f"  [skip] n={n}: no usable rows")
        continue

    Xa = pd.concat([X_train_7.reset_index(drop=True), Xs], axis=0).reset_index(drop=True)
    ya = pd.concat([y_train_7.reset_index(drop=True), ys], axis=0).reset_index(drop=True)
    new_share = 100 * (ya == -3).mean()

    clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(Xa, ya)
    y_pred = clf.predict(X_test_7)

    log_result(f"MostlyAI Cold-conditional (n={n:,})", "Augmented", 7,
               y_test_7, y_pred,
               notes=f"Cold share after augmentation: {new_share:.2f}%")

    cm = confusion_matrix(y_test_7, y_pred, labels=sorted(pd.unique(y_test_7)))
    cold_row = cm[0]
    print(f"    Cold share after augmentation: {new_share:.2f}%")
    print(f"    True Cold predicted as: "
          + ", ".join(f"{lb}:{c}" for lb, c in
                      zip(sorted(pd.unique(y_test_7)), cold_row) if c > 0))
    print()

CONDITION B — conditional Cold augmentation

  MostlyAI Cold-conditional (n=10,000) Augmented  7-class  macro_f1=0.2059  cold_f1=0.2161
    Cold share after augmentation: 4.51%
    True Cold predicted as: -3:19336, -2:2531, -1:689

  MostlyAI Cold-conditional (n=50,000) Augmented  7-class  macro_f1=0.1624  cold_f1=0.1826
    Cold share after augmentation: 7.39%
    True Cold predicted as: -3:22451, -2:105

  MostlyAI Cold-conditional (n=200,000) Augmented  7-class  macro_f1=0.0650  cold_f1=0.1513
    Cold share after augmentation: 16.80%
    True Cold predicted as: -3:22556



## Part 6 — Save everything

The CSV is now the authoritative source for your Results chapter. Never copy a
number into your thesis from anywhere else.

In [12]:
res = pd.DataFrame(RESULTS)
res["delta_vs_baseline"] = res.apply(
    lambda r: round(r["macro_f1"] - (BASE_7 if r["granularity"] == 7 else BASE_3), 4),
    axis=1)
res["retention_pct"] = res.apply(
    lambda r: round(100 * r["macro_f1"] / (BASE_7 if r["granularity"] == 7 else BASE_3), 1),
    axis=1)

csv_path = f"results/wk10_mostlyai_results.csv"
res.to_csv(csv_path, index=False)

MANIFEST["finished"] = datetime.now().isoformat(timespec="seconds")
MANIFEST["n_conditions_logged"] = len(res)
with open("results/wk10_manifest.json", "w") as f:
    json.dump(MANIFEST, f, indent=2)

cols = ["method", "protocol", "granularity", "macro_f1",
        "delta_vs_baseline", "retention_pct", "cold_f1"]
print(res[cols].to_string(index=False))
print(f"\nSaved: {csv_path}")
print("Saved: results/wk10_manifest.json")

print("\n" + "=" * 68)
print("HOW TO READ THIS")
print("=" * 68)
print("""
delta_vs_baseline < 0 for every row
    -> no method beat the baseline. Report that plainly. It is the finding.

cold_f1 = 0.0000 even at n=200,000 conditional Cold rows
    -> the strong version of your claim now HOLDS, and you have tested it.
       You may write that targeted minority synthesis at 200k rows still
       recovers nothing.

cold_f1 > 0 at some n
    -> your thesis needs revising. That is a real result and worth having.
       Report the n at which it becomes non-zero and what it costs in macro-F1.
""")

                               method  protocol  granularity  macro_f1  delta_vs_baseline  retention_pct  cold_f1
                 Baseline (real only)      TRTR            7    0.2858             0.0000          100.0   0.0000
                 Baseline (real only)      TRTR            3    0.7163             0.0000          100.0   0.7120
                             MostlyAI      TSTR            7    0.2119            -0.0739           74.1   0.0000
                             MostlyAI Augmented            7    0.2563            -0.0295           89.7   0.0000
                             MostlyAI      TSTR            3    0.6601            -0.0562           92.2   0.7341
                             MostlyAI Augmented            3    0.6719            -0.0444           93.8   0.7440
 MostlyAI Cold-conditional (n=10,000) Augmented            7    0.2059            -0.0799           72.0   0.2161
 MostlyAI Cold-conditional (n=50,000) Augmented            7    0.1624            -0.123

## Part 7

In [14]:
ref = res[res["method"].str.contains("MostlyAI|Baseline")]
piv = ref.pivot_table(index=["method", "protocol"], columns="granularity",
                      values=["macro_f1", "cold_f1"], aggfunc="first")

print("=" * 68)
print(f"Source: {csv_path}   Run ID: {RUN_ID}")
print(piv.to_string())

Source: results/wk10_mostlyai_results.csv   Run ID: 20260804_231218
                                                cold_f1         macro_f1        
granularity                                           3       7        3       7
method                                protocol                                  
Baseline (real only)                  TRTR       0.7120  0.0000   0.7163  0.2858
MostlyAI                              Augmented  0.7440  0.0000   0.6719  0.2563
                                      TSTR       0.7341  0.0000   0.6601  0.2119
MostlyAI Cold-conditional (n=10,000)  Augmented     NaN  0.2161      NaN  0.2059
MostlyAI Cold-conditional (n=200,000) Augmented     NaN  0.1513      NaN  0.0650
MostlyAI Cold-conditional (n=50,000)  Augmented     NaN  0.1826      NaN  0.1624


## Outcome of this run (Run ID 20260804_231218)

Conditional Cold generation produced **non-zero Cold F1**, contradicting the
assumption carried through wk1 to wk9 that Cold F1 is an immovable zero.

| n synthetic Cold | Cold F1 | Macro F1 | Recall | Precision | % of test set predicted Cold |
|---|---|---|---|---|---|
| 10,000  | 0.2161 | 0.2059 | 0.857 | 0.124 | 53.9% |
| 50,000  | 0.1826 | 0.1624 | 0.995 | 0.101 | 77.0% |
| 200,000 | 0.1513 | 0.0650 | 1.000 | 0.082 | 95.0% |

Test-set Cold prevalence is 7.78% (22,556 of 290,019 rows). A degenerate
classifier predicting Cold for every row would score Cold F1 = 0.1443. The
n=200,000 result (0.1513) is therefore barely distinguishable from that
degenerate predictor, and indeed 95% of the test set was predicted Cold.
Precision falls toward base prevalence as synthetic volume rises, while macro
F1 collapses from 0.2858 to 0.0650.

**Interpretation carried into the thesis:** targeted minority synthesis shifts
the decision prior rather than improving discrimination. More synthetic Cold
data makes Cold F1 worse, not better. This is consistent with the prior-shift
collapse observed under Gaussian Copula full class balancing in wk2
(macro F1 0.120).

### Known issues with this run

1. **MostlyAI does not reproduce.** 3-class figures here are TSTR 0.6601 and
   Augmented 0.6719, against wk4's 0.7003 and 0.7066. The Random Forest
   baselines reproduce wk4 exactly (0.2858, 0.7163), which localises the
   non-determinism to the generator rather than the data pipeline. The
   generator was not seeded.

2. **The closing "HOW TO READ THIS" block below is stale.** It was drafted
   before the run and states that a zero Cold F1 confirms the thesis claim.
   The observed Cold F1 was not zero. Read the results table, not that block.

3. **3-class Cold F1 is not zero.** The 3-class baseline scores 0.7120 on
   Cold, because the 3-class "Cold" class merges labels -3 and -2 and is
   learnable. Any claim of zero Cold F1 applies only to the 7-class label -3.